In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

In [48]:
df=pd.read_csv('Churn_Modelling.csv')

In [49]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [50]:
df.drop(columns=['RowNumber','CustomerId','Surname'],inplace=True)

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  object 
 2   Gender           10000 non-null  object 
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


In [52]:
df['Geography'].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [53]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=['Exited']),df['Exited'],test_size=0.2,random_state=2)
x_train

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
8856,774,France,Female,26,5,0.00,2,1,1,64716.08
1545,602,Germany,Female,42,6,158414.85,1,1,1,131886.46
8136,716,France,Female,38,1,0.00,2,1,1,99661.46
7768,601,Spain,Female,20,9,122446.61,2,1,0,86791.90
7782,585,France,Male,54,8,87105.32,1,1,1,55346.14
...,...,...,...,...,...,...,...,...,...,...
1099,494,Spain,Male,38,7,0.00,2,1,1,6203.66
2514,592,Germany,Female,26,4,105082.07,2,1,0,132801.57
6637,850,France,Male,36,3,0.00,2,1,0,195033.07
2575,551,Germany,Male,54,5,102994.04,1,1,0,176680.16


In [54]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
ct=ColumnTransformer(
    transformers=[
        ('cat',OneHotEncoder(drop='first',sparse_output=False),['Geography','Gender']),
        ('num',StandardScaler(),['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary'])
    ],remainder='passthrough')
model=LogisticRegression(
    max_iter=1000,class_weight='balanced',random_state=42)
pipeline=Pipeline(steps=[
    ('transformer',ct),
    ('model',model)
])
pipeline.fit(x_train,y_train)

,steps,"[('transformer', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [55]:
from sklearn.metrics import(accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report)
y_pred=pipeline.predict(x_test)
y_prob=pipeline.predict_proba(x_test)[:,1]
print("acc:",accuracy_score(y_test,y_pred),4)
print('precision:',precision_score(y_test,y_pred),4)
print('recall:',recall_score(y_test,y_pred),4)
print('f1:',f1_score(y_test,y_pred),4)
print('roc-auc:',roc_auc_score(y_test,y_prob),4)
print('confusion_matrix:',confusion_matrix(y_test,y_pred))
print('classification_report:',classification_report(y_test,y_pred))

acc: 0.707 4
precision: 0.35684062059238364 4
recall: 0.660574412532637 4
f1: 0.4633699633699634 4
roc-auc: 0.7595424592813627 4
confusion_matrix: [[1161  456]
 [ 130  253]]
classification_report:               precision    recall  f1-score   support

           0       0.90      0.72      0.80      1617
           1       0.36      0.66      0.46       383

    accuracy                           0.71      2000
   macro avg       0.63      0.69      0.63      2000
weighted avg       0.80      0.71      0.73      2000



In [56]:
import pickle
pickle.dump(pipeline,open('Churn_model.pkl','wb'))

In [57]:
with open('Churn_model.pkl','rb') as file:
    loaded_model=pickle.load(file)
print('model loaded successfully')

model loaded successfully


In [58]:
new_customer=x_test.iloc[[0]]
prediction=loaded_model.predict(new_customer)[0]
probability=loaded_model.predict_proba(new_customer)[0,1]
print("predicted:",prediction)
print("churn probability:",round(probability,3))

predicted: 1
churn probability: 0.537
